# Cross-Modal Knowledge Distillation: ConcatMTLFaceRecognitionV2 → MobileNetV3

**Mục tiêu:** Teacher (albedo + normalmap) truyền tri thức đa modal cho student chỉ dùng **một modality duy nhất** lúc inference.

| | Teacher | Student |
|---|---|---|
| Model | `ConcatMTLFaceRecognitionV2` (ConvNeXt V2 × 2) | `FaceRecognitionMobileNetV3` |
| Input | albedo **+** normalmap `[B, 2, 3, 112, 112]` | albedo **hoặc** normalmap `[B, 3, 112, 112]` |
| ID Embedding | 1024-D (512 albedo ⊕ 512 normalmap) | 512-D |
| Mode | **Frozen** | **Trainable** |

**KD Loss:**
```
L_total = α · L_MagFace(student)  +  β · L_KD   + γ·L_RKD_D  +  δ·L_RKD_A
L_MagFace   : WeightClassMagLoss  — phân biệt class boundary cho student
L_KD    = mean(1 - cosine_sim(norm(proj(s_emb)), norm(t_emb)))
L_RKD_D = Huber( dist_s(i,j)/μ_s − dist_t(i,j)/μ_t )   — bảo toàn khoảng cách tương đối
L_RKD_A = Huber( cos∠_s(i,j,k) − cos∠_t(i,j,k) )       — bảo toàn góc giữa bộ ba
```

**ProjectionHead:** `Linear(512→1024) → BN1d → ReLU → Linear(1024→1024)` — bridge student (512-D) sang teacher space (1024-D), chỉ dùng khi train.

**Cách dùng:**
1. Chạy cell 1 (Mount Drive)
2. Sửa `CONFIGURATION`, `TEACHER_CKPT_1`, `TEACHER_CKPT_2`, `STUDENT_MODAL_IDX`
3. Chạy từ trên xuống

## 1. Mount Drive & Setup môi trường

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_URL    = 'https://github.com/NguyenXuanBinh22/DATN.git'
REPO_BRANCH = 'convnext-v2-dev'
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}')
    print('Repo đã tồn tại, đã pull latest.')

%cd {REPO_DIR}
print(f'Working dir: {os.getcwd()}')

os.system('pip install -q albumentations==1.3.1 timm tabulate termcolor onnx')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo đã tồn tại, đã pull latest.
/content/FR_Photometric_Stereo
Working dir: /content/FR_Photometric_Stereo


0

## 2. Imports & Cấu hình

In [ ]:
%cd /content/FR_Photometric_Stereo
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import albumentations as A
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.tensorboard import SummaryWriter
from tabulate import tabulate
import onnx

from going_modular.dataloader.multitask import create_concatv2_multitask_datafetcher, create_eval_loaders
from going_modular.model.MTLFaceRecognition import MTLFaceRecognition
from going_modular.model.ConcatMTLFaceRecognition import ConcatMTLFaceRecognitionV2
from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3
from going_modular.loss.WeightClassMagLoss import WeightClassMagLoss
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.roc_auc_id import (
    compute_id_auc, compute_rank1,
    compute_id_auc_gallery_probe, compute_rank1_gallery_probe,
)
from going_modular.utils.MultiMetricEarlyStopping import MultiMetricEarlyStopping
from going_modular.utils.ModelCheckPoint import ModelCheckpoint
from going_modular.utils.ExperimentManager import ExperimentManager

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

/content/FR_Photometric_Stereo
Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [ ]:
# ════════════════════════════════════════════════════════════
#  CẤU HÌNH — chỉnh sửa ở đây
# ════════════════════════════════════════════════════════════

DRIVE_DATASET_DIR = '/content/drive/MyDrive/Photometric_DB_Full/'

# Checkpoint của 2 single-modal teacher đã train xong
TEACHER_CKPT_1 = '/content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'
TEACHER_CKPT_2 = '/content/drive/MyDrive/Photometric_DB_Full/experiments/Single_NORMALMAP_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'

# Modality student nhận lúc inference: 0 = albedo (backbone1), 1 = normalmap (backbone2)
STUDENT_MODAL_IDX = 0

EXPERIMENT_NAME = '(1-15-100-200)new_lan2_KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo'

CONFIGURATION = {
    'note':        EXPERIMENT_NAME,
    'dataset_dir': DRIVE_DATASET_DIR,
    'output_dir':  '/content/drive/MyDrive/',

    # concat_v2: dataloader trả [B, 2, 3, H, W] (albedo=[:,0], normalmap=[:,1])
    'type':        'concat_v2',

    'teacher_backbone': 'convnextv2_tiny',
    'backbone':         'mobilenetv3_large_100',
    'use_sampler': True,
    'device':      device,
    'epochs':      100,
    'num_workers': 2,
    'batch_size':  16,
    'image_size':  112,
    'base_lr':     1e-4,
    'num_classes': None,

    # L_total = task_weight * L_MagFace + kd_weight * L_KD + rkd_d_weight * L_RKD_D + rkd_a_weight * L_RKD_A
    'task_weight': 1.0,
    'kd_weight':   15,
    'rkd_d_weight': 100.0,
    'rkd_a_weight': 200.0,

}

modal_name = ['albedo', 'normalmap'][STUDENT_MODAL_IDX]
print(f'Student sẽ học từ modal: {modal_name} (index {STUDENT_MODAL_IDX})')
print(f'Teacher ckpt 1 (albedo) : {TEACHER_CKPT_1}')
print(f'Teacher ckpt 2 (normal) : {TEACHER_CKPT_2}')

Student sẽ học từ modal: albedo (index 0)
Teacher ckpt 1 (albedo) : /content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth
Teacher ckpt 2 (normal) : /content/drive/MyDrive/Photometric_DB_Full/experiments/Single_NORMALMAP_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth


## 3. Data Loading

`create_concatv2_multitask_datafetcher` trả batch `[B, 2, 3, H, W]`:
- `X[:, 0]` = albedo → teacher backbone1 + student input (nếu `STUDENT_MODAL_IDX=0`)
- `X[:, 1]` = normalmap → teacher backbone2 + student input (nếu `STUDENT_MODAL_IDX=1`)

Teacher dùng cả 2, student chỉ dùng `X[:, STUDENT_MODAL_IDX]`.

In [ ]:
dataset_dir = CONFIGURATION['dataset_dir']

train_csv = os.path.join(dataset_dir, 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'dataset', 'train_split.csv')
if not os.path.exists(train_csv):
    raise FileNotFoundError(f'Không tìm thấy CSV train tại {dataset_dir}.')
print(f'Train CSV: {train_csv}')

df_train = pd.read_csv(train_csv)
# CONFIGURATION['num_classes'] = int(df_train['id'].nunique())
CONFIGURATION['num_classes'] = 2232
print(f'num_classes : {CONFIGURATION["num_classes"]}')
print(f'Số mẫu train: {len(df_train)}')

# additional_targets={'image2': 'image'} vì concat dataloader dùng key 'image2' cho modal thứ 2
train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
], additional_targets={'image2': 'image'})

test_transform = A.Compose([
    A.Resize(height=CONFIGURATION['image_size'], width=CONFIGURATION['image_size']),
], additional_targets={'image2': 'image'})

train_dl, test_dl = create_concatv2_multitask_datafetcher(
    CONFIGURATION, train_transform, test_transform, 'train_split.csv', 'probe_split.csv'
)
print(f'Train batches: {len(train_dl)} | Test batches (probe): {len(test_dl)}')

# Kiểm tra shape batch
X_sample, y_sample = next(iter(train_dl))
print(f'Batch shape: X={X_sample.shape}, y={y_sample.shape}')  # [B, 2, 3, 112, 112]

# Gallery-probe loader cho đánh giá cuối
# Cần tạo eval loader cho đúng modality mà student dùng
eval_conf = dict(CONFIGURATION)
eval_conf['type'] = ['albedo', 'normalmap'][STUDENT_MODAL_IDX]
gallery_dl, probe_dl = create_eval_loaders(eval_conf, A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size'])
]))
print(f'Gallery batches: {len(gallery_dl)} | Probe batches: {len(probe_dl)}')

Train CSV: /content/drive/MyDrive/Photometric_DB_Full/train_split.csv
num_classes : 2232
Số mẫu train: 2622
>>> ConcatV2Loader: MODE = PK SAMPLER
Train batches: 163 | Test batches (probe): 18
Batch shape: X=torch.Size([16, 2, 3, 112, 112]), y=torch.Size([16, 6])
Gallery: 68 ảnh | Probe: 288 ảnh
Shared identity space: 68 identities
Gallery batches: 5 | Probe batches: 18


## 4. Teacher Model (ConcatMTLFaceRecognitionV2 — Frozen)

Load 2 single-modal checkpoint đã train, ghép thành `ConcatMTLFaceRecognitionV2`, freeze toàn bộ.

`get_result(X)[0]` trả `id_embedding` **1024-D** (concat từ 2 backbone 512-D mỗi cái).

In [ ]:
for path, name in [(TEACHER_CKPT_1, 'albedo'), (TEACHER_CKPT_2, 'normalmap')]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Không tìm thấy checkpoint {name}: {path}')

def _load_mtl_backbone(ckpt_path, backbone, num_classes, device):
    model = MTLFaceRecognition(backbone=backbone, num_classes=num_classes)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    state_dict = ckpt['model_state_dict']
    keys_to_remove = [k for k in state_dict.keys() if 'id_head.maglinear' in k]
    for k in keys_to_remove:
      del state_dict[k]
    model.load_state_dict(state_dict, strict=False)
    print(f'  Loaded: {ckpt_path} (epoch {ckpt.get("epoch", "?")})')
    return model

print('Loading backbone 1 (albedo)...')
mtl_backbone1 = _load_mtl_backbone(
    TEACHER_CKPT_1, CONFIGURATION['teacher_backbone'], CONFIGURATION['num_classes'], device
)
mtl_backbone1.to(device)
mtl_backbone1.eval()

for p in mtl_backbone1.parameters():
    p.requires_grad = False

mtl_01_params = sum(p.numel() for p in mtl_backbone1.parameters())
print(f'\MTL backbone 1 params: {mtl_01_params:,} (tất cả frozen)')

print('Loading backbone 2 (normalmap)...')
mtl_backbone2 = _load_mtl_backbone(
    TEACHER_CKPT_2, CONFIGURATION['teacher_backbone'], CONFIGURATION['num_classes'], device
)

teacher = ConcatMTLFaceRecognitionV2(mtl_backbone1, mtl_backbone2, CONFIGURATION['num_classes'])
teacher.to(device)
teacher.eval()

# Freeze hoàn toàn
for p in teacher.parameters():
    p.requires_grad = False

teacher_params = sum(p.numel() for p in teacher.parameters())
print(f'\nTeacher params: {teacher_params:,} (tất cả frozen)')

# Smoke test — kiểm tra embedding shape
with torch.no_grad():
    _dummy = torch.randn(2, 2, 3, 112, 112).to(device)  # [B, 2, 3, H, W]
    _t_emb = teacher.get_result(_dummy)[0]               # id_embedding: [B, 1024]
    print(f'Teacher ID embedding shape: {_t_emb.shape}')  # mong đợi [2, 1024]

with torch.no_grad():
  _dummy = torch.randn(2,3,112,112).to(device)
  _t_emb = mtl_backbone1.get_result(_dummy)[0]
  print(f'MTL Backbone 1 ID embedding shape: {_t_emb.shape}')

Loading backbone 1 (albedo)...


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

  Loaded: /content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth (epoch 28)
\MTL backbone 1 params: 45,670,280 (tất cả frozen)
Loading backbone 2 (normalmap)...


  Loaded: /content/drive/MyDrive/Photometric_DB_Full/experiments/Single_NORMALMAP_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth (epoch 96)

Teacher params: 91,711,258 (tất cả frozen)
Teacher ID embedding shape: torch.Size([2, 1024])
MTL Backbone 1 ID embedding shape: torch.Size([2, 512])


## 4.1. Đánh giá Teacher (baseline)

Đo AUC của teacher trên gallery-probe để có baseline so sánh.

In [ ]:
# modal_name = ['albedo', 'normalmap'][STUDENT_MODAL_IDX]

# # ── Baseline 1: ConvNeXt single-modal (cùng modality với student) ──────────────
# class _TeacherSingleModalWrapper(nn.Module):
#     """Chỉ dùng 1 backbone của teacher — baseline công bằng với student."""
#     def __init__(self, teacher, modal_idx):
#         super().__init__()
#         self._backbone = teacher.mtl_backbone1 if modal_idx == 0 else teacher.mtl_backbone2

#     def get_embedding(self, x):
#         return self._backbone.get_embedding(x)[-1]  # [B, 512]


# teacher_single = _TeacherSingleModalWrapper(teacher, STUDENT_MODAL_IDX).to(device)
# teacher_single.eval()

# teacher_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, teacher_single, device)
# teacher_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, teacher_single, device)
# print(f'Baseline 1 (single-modal {modal_name}) done.')

# # ── Baseline 2: ConcatTeacher full (2 modal) — upper bound ─────────────────────
# class _TeacherConcatWrapper(nn.Module):
#     """Wrap ConcatMTLFaceRecognitionV2, expose get_embedding() nhận [B, 2, 3, H, W]."""
#     def __init__(self, teacher):
#         super().__init__()
#         self._teacher = teacher

#     def get_embedding(self, x):
#         # get_result() → (id_embedding 1024-D, gender, pose, emotion, facial_hair, spectacles)
#         return self._teacher.get_result(x)[0]  # [B, 1024]


# # Gallery/probe loader trả [B, 2, 3, H, W] cho full teacher
# concat_eval_conf = dict(CONFIGURATION)
# concat_eval_conf['type'] = 'concat_v2'
# gallery_concat_dl, probe_concat_dl = create_eval_loaders(
#     concat_eval_conf,
#     A.Compose([
#         A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size']),
#     ], additional_targets={'image2': 'image'})
# )

# teacher_concat = _TeacherConcatWrapper(teacher).to(device)
# teacher_concat.eval()

# teacher_concat_auc   = compute_id_auc_gallery_probe(gallery_concat_dl, probe_concat_dl, teacher_concat, device)
# teacher_concat_rank1 = compute_rank1_gallery_probe(gallery_concat_dl, probe_concat_dl, teacher_concat, device)
# print(f'Baseline 2 (full ConcatTeacher, 2 modal) done.')

# # ── Bảng tổng hợp 2 baseline ────────────────────────────────────────────────────
# compare_rows = [
#     ['Input',         f'{modal_name} only',                'albedo + normalmap'],
#     ['Embedding dim', '512-D',                             '1024-D'],
#     ['Cosine AUC    (gallery→probe)',
#      f"{teacher_gp_auc['id_cosine']:.4f}",
#      f"{teacher_concat_auc['id_cosine']:.4f}"],
#     ['Euclidean AUC (gallery→probe)',
#      f"{teacher_gp_auc['id_euclidean']:.4f}",
#      f"{teacher_concat_auc['id_euclidean']:.4f}"],
#     ['Rank-1 Acc    (gallery→probe)',
#      f"{teacher_gp_rank1:.4f}",
#      f"{teacher_concat_rank1:.4f}"],
# ]
# print(f"\n--- Teacher Baselines ({CONFIGURATION['teacher_backbone']}) ---")
# print(tabulate(compare_rows,
#                headers=['Metric', 'Single-modal (baseline)', 'Full teacher (upper bound)'],
#                tablefmt='fancy_grid'))

## 5. Student Model (MobileNetV3 — Trainable)

Nhận **1 modality** `[B, 3, 112, 112]`, output embedding **512-D**.

In [ ]:
student = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
)
student.to(device)

total_p     = sum(p.numel() for p in student.parameters())
trainable_p = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'Student total params    : {total_p:,}')
print(f'Student trainable params: {trainable_p:,}')

# Smoke test
_dummy_single = torch.randn(2, 3, 112, 112).to(device)
with torch.no_grad():
    _s_emb = student.get_embedding(_dummy_single)
    print(f'Student embedding shape: {_s_emb.shape}')  # mong đợi [2, 512]

model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

Student total params    : 3,645,744
Student trainable params: 3,645,744
Student embedding shape: torch.Size([2, 512])


In [ ]:
# ── Kiến trúc Student ────────────────────────────────────────────────────────
print('=' * 60)
print('STUDENT MODEL ARCHITECTURE')
print('=' * 60)
print(student)
print()

# ── Thống kê params từng component ──────────────────────────────────────────
def count_params(module):
    total     = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable

rows = []
for name, module in [('backbone',  student.backbone),
                     ('embedding', student.embedding),
                     ('maglinear', student.maglinear)]:
    total, trainable = count_params(module)
    rows.append([name, f'{total:,}', f'{trainable:,}'])

total_all,     trainable_all     = count_params(student)
rows.append(['─' * 10, '─' * 12, '─' * 12])
rows.append(['TOTAL', f'{total_all:,}', f'{trainable_all:,}'])

print(tabulate(rows, headers=['Component', 'Params', 'Trainable'], tablefmt='fancy_grid'))

STUDENT MODEL ARCHITECTURE
FaceRecognitionMobileNetV3(
  (backbone): MIMobileNetV3(
    (backbone): MobileNetV3Features(
      (conv_stem): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act1): Hardswish()
      (blocks): Sequential(
        (0): Sequential(
          (0): DepthwiseSeparableConv(
            (conv_dw): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
            (bn1): BatchNormAct2d(
              16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
              (drop): Identity()
              (act): ReLU(inplace=True)
            )
            (aa): Identity()
            (se): Identity()
            (conv_pw): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (bn2): BatchNormAct2d(
              16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=Tru

## 6. Projection Head (512-D → 1024-D)

Bridge student space (512-D) sang teacher space (1024-D) để tính KD loss.

```
student_emb (512) → Linear(512→1024) → BN1d → ReLU → Linear(1024→1024) → proj_emb (1024)
                                                                              ↕ cosine KD loss
                                                             teacher_id_emb (1024)
```

- `student_emb` vẫn đi thẳng vào `MagLinear` (task loss không thay đổi)
- Projector **chỉ tồn tại khi train**, bỏ hoàn toàn khi export ONNX

In [ ]:
class ProjectionHead(nn.Module):
    """
    MLP bridge: student embedding space (512) → teacher embedding space (1024).
    Chỉ dùng khi tính KD loss, không tham gia inference / ONNX export.
    """
    def __init__(self, in_dim: int = 512, hidden_dim: int = 1024, out_dim: int = 1024):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)


projector = ProjectionHead(in_dim=512, hidden_dim=1024, out_dim=1024).to(device)
proj_params = sum(p.numel() for p in projector.parameters())
print(f'ProjectionHead params: {proj_params:,}  (train-only, dropped at export)')

# Smoke test
with torch.no_grad():
    _proj_emb = projector(_s_emb)
    print(f'Projected embedding shape: {_proj_emb.shape}')  # mong đợi [2, 1024]

ProjectionHead params: 1,576,960  (train-only, dropped at export)
Projected embedding shape: torch.Size([2, 1024])


## 7. Knowledge Distillation Loss

```
L_total = α·L_MagFace(student_emb)  
+  β·L_KD_cosine(proj(student_emb), teacher_emb)  
+  γ·L_RKD_D  
+  δ·L_RKD_A
```

**Tại sao cosine loss:** face verification so sánh *hướng* embedding, không phải magnitude.  
Cosine loss kéo student về đúng hướng mà teacher biết từ thông tin đa modal.

In [ ]:
class newCrossModalKDLoss(nn.Module):
    """
    L_total = task_w*L_MagFace
            + kd_w*L_KD_cosine(proj_emb, teacher_emb)
            + rkd_d_w*L_RKD_D(student_emb, mtl_1_teacher_emb)
            + rkd_a_w*L_RKD_A(student_emb, mtl_1_teacher_emb)

    L_KD_cosine dùng proj_emb = projector(student_emb) thay vì student_emb trực tiếp.
    RKD-D/A vẫn dùng student_emb gốc để bảo toàn relational structure thực sự.
    student_emb  : [B, 512]  — embedding trước MagLinear (task loss)
    proj_emb     : [B, 1024] — student_emb sau ProjectionHead (KD loss)
    teacher_emb  : [B, 1024] — fused ID embedding của ConcatTeacher (no_grad)
    """

    def __init__(
        self,
        metadata_path: str,
        task_weight:   float = 1.0,
        kd_weight:     float = 1.0,
        rkd_d_weight:  float = 80.0,
        rkd_a_weight:  float = 160.0,
      ):
        super().__init__()
        self.magface = WeightClassMagLoss(metadata_path)
        self.task_w  = task_weight
        self.kd_w    = kd_weight
        self.rkd_d_w  = rkd_d_weight
        self.rkd_a_w  = rkd_a_weight

    @staticmethod
    def _pdist(e: torch.Tensor) -> torch.Tensor:
        diff = e.unsqueeze(0) - e.unsqueeze(1)
        return diff.pow(2).sum(-1).clamp(min=1e-12).sqrt()

    def _rkd_distance(self, s_emb: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        B = s_emb.size(0)
        mask = ~torch.eye(B, dtype=torch.bool, device=s_emb.device)

        with torch.no_grad():
            td   = self._pdist(t_emb)
            mu_t = td[mask].mean()
            td_n = td / (mu_t + 1e-8)

        sd   = self._pdist(s_emb)
        mu_s = sd[mask].mean()
        sd_n = sd / (mu_s + 1e-8)

        return F.huber_loss(sd_n[mask], td_n[mask], delta=1.0, reduction='mean')

    def _rkd_angle(self, s_emb: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        B       = s_emb.size(0)
        eye     = torch.eye(B, dtype=torch.bool, device=s_emb.device)
        mask_3d = (~eye.unsqueeze(2)) & (~eye.unsqueeze(1))

        def _angle_matrix(e: torch.Tensor) -> torch.Tensor:
            diff = e.unsqueeze(0) - e.unsqueeze(1)
            norm = diff.norm(p=2, dim=2, keepdim=True).clamp(min=1e-8)
            diff = diff / norm
            return torch.bmm(diff, diff.transpose(1, 2))

        with torch.no_grad():
            ta = _angle_matrix(t_emb)
        sa = _angle_matrix(s_emb)

        return F.huber_loss(sa[mask_3d], ta[mask_3d], delta=1.0, reduction='mean')


    def forward(
        self,
        student_logits,
        student_norm,
        student_emb,
        proj_emb,
        teacher_emb,
        mtl_1_teacher_emb,
        id_labels,
    ):
        l_task = self.magface(student_logits, id_labels, student_norm)

        # KD loss trên unit hypersphere 1024-D
        p_n = F.normalize(proj_emb,    p=2, dim=1)
        t_n = F.normalize(teacher_emb, p=2, dim=1)
        l_kd = (1.0 - F.cosine_similarity(p_n, t_n, dim=1)).mean()

        # RKD: dùng student_emb gốc để bảo toàn relational structure thực sự
        l_rkd_d = self._rkd_distance(student_emb, mtl_1_teacher_emb)
        l_rkd_a = self._rkd_angle(student_emb, mtl_1_teacher_emb)

        total = (
            self.task_w  * l_task
          + self.kd_w    * l_kd
          + self.rkd_d_w * l_rkd_d
          + self.rkd_a_w * l_rkd_a
        )
        return total, l_task, l_kd, l_rkd_d, l_rkd_a


criterion = newCrossModalKDLoss(
    metadata_path=train_csv,
    task_weight=CONFIGURATION['task_weight'],
    kd_weight=CONFIGURATION['kd_weight'],
    rkd_d_weight=CONFIGURATION['rkd_d_weight'],
    rkd_a_weight=CONFIGURATION['rkd_a_weight'],
)

print('newCrossModalKDLoss khởi tạo thành công.')
print(f"  task={CONFIGURATION['task_weight']}  "
      f"kd={CONFIGURATION['kd_weight']} (qua projector)  "
      f"rkd_d={CONFIGURATION['rkd_d_weight']}  "
      f"rkd_a={CONFIGURATION['rkd_a_weight']}")

newCrossModalKDLoss khởi tạo thành công.
  task=1.0  kd=15 (qua projector)  rkd_d=100.0  rkd_a=200.0


## 8. Training

In [ ]:
def train_epoch(train_dl, teacher, mtl_backbone1, student, projector, criterion, optimizer, device, student_modal_idx):
    student.train()
    projector.train()
    # teacher.eval() + frozen — không cần đặt lại mỗi epoch

    total_loss = total_task = total_kd = total_rkd_d = total_rkd_a = 0.0

    for X, y in train_dl:
        # X shape: [B, 2, 3, H, W]  (axis 1: 0=albedo, 1=normalmap)
        X, y = X.to(device), y.to(device)
        id_labels = y[:, 0]

        # Teacher nhận cả 2 modality — lấy fused ID embedding 1024-D
        with torch.no_grad():
            teacher_emb = teacher.get_result(X)[0]  # [B, 1024]

        # Student chỉ nhận 1 modality
        X_student = X[:, student_modal_idx]  # [B, 3, H, W]

        with torch.no_grad():
            mtl_1_teacher_emb = mtl_backbone1.get_embedding(X_student)[-1]  # [B, 512], frozen



        feat        = student.backbone(X_student)     # [B, 512, H', W']
        student_emb = student.embedding(feat)         # [B, 512]
        proj_emb    = projector(student_emb)          # [B, 1024] — bridge to teacher space
        logits, norm = student.maglinear(student_emb)
# warning!!!
        loss, l_task, l_kd, l_rkd_d, l_rkd_a = criterion(
            logits, norm, student_emb, proj_emb, teacher_emb, mtl_1_teacher_emb, id_labels
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_task += l_task.item()
        total_kd   += l_kd.item()
        total_rkd_d += l_rkd_d.item()
        total_rkd_a += l_rkd_a.item()

    n = len(train_dl)
    return (
        total_loss  / n,
        total_task  / n,
        total_kd    / n,
        total_rkd_d / n,
        total_rkd_a / n,
    )


def display_metrics(epoch, train_metrics, test_metrics):
    rows = []
    for k in train_metrics:
        tv = train_metrics[k]
        ev = test_metrics.get(k, '-')
        fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
        rows.append([k, fmt(tv), fmt(ev)])
    print(f'\nEp {epoch}:')
    print(tabulate(rows, headers=['Metric', 'Train', 'Test'], tablefmt='fancy_grid'))

In [ ]:
# Optimizer update cả student lẫn projector
optimizer = Adam(
    list(student.parameters()) + list(projector.parameters()),
    lr=CONFIGURATION['base_lr'],
)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

manager = ExperimentManager(CONFIGURATION)
manager.log_text(
    f"Teacher: ConcatMTLFaceRecognitionV2 ({CONFIGURATION['teacher_backbone']} x2) | "
    f"Student: {CONFIGURATION['backbone']} | "
    f"Student modal: {['albedo','normalmap'][STUDENT_MODAL_IDX]} (idx={STUDENT_MODAL_IDX}) | "
    f"Teacher emb: 1024-D | Student emb: 512-D | "
    f"task={CONFIGURATION['task_weight']} kd={CONFIGURATION['kd_weight']} (projector) |"
    f"rkd_d={CONFIGURATION['rkd_d_weight']} rkd_a={CONFIGURATION['rkd_a_weight']} | "
    f"projector=True (512→1024)"
)

ckpt_saver = ModelCheckpoint(
    output_dir=manager.ckpt_dir,
    mode='max',
    best_metric_name='auc_id_cosine',
)
early_stopping = MultiMetricEarlyStopping(
    monitor_keys=['auc_id_cosine'],
    patience=10,
    mode='max',
    verbose=1,
    save_dir=manager.ckpt_dir,
    start_from_epoch=5,
)

writer = SummaryWriter(log_dir=manager.log_dir)
print(f'Experiment dir: {manager.exp_dir}')
print(f'Checkpoint dir: {manager.ckpt_dir}')

KHOI TAO THI NGHIEM: (1-15-100-200)new_lan2_KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo
Luu tru tai: /content/drive/MyDrive/experiments/(1-15-100-200)new_lan2_KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo
Thoi gian: 2026-06-12 10:19:26
--------------------------------------------------
Teacher: ConcatMTLFaceRecognitionV2 (convnextv2_tiny x2) | Student: mobilenetv3_large_100 | Student modal: albedo (idx=0) | Teacher emb: 1024-D | Student emb: 512-D | task=1.0 kd=15 (projector) |rkd_d=100.0 rkd_a=200.0 | projector=True (512→1024)
Experiment dir: /content/drive/MyDrive/experiments/(1-15-100-200)new_lan2_KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo
Checkpoint dir: /content/drive/MyDrive/experiments/(1-15-100-200)new_lan2_KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo/checkpoints


In [ ]:
START_EPOCH = 0

manager.log_text('BAT DAU new CROSS-MODAL KNOWLEDGE DISTILLATION')

for epoch in range(START_EPOCH, CONFIGURATION['epochs']):
    manager.log_text(f'\n--- Epoch {epoch+1}/{CONFIGURATION["epochs"]} ---')

    train_loss, train_task, train_kd, train_rkd_d, train_rkd_a = train_epoch(
        train_dl, teacher, mtl_backbone1, student, projector, criterion,
        optimizer, device, STUDENT_MODAL_IDX,
    )

    # AUC dùng student.get_embedding() trực tiếp (không qua projector)
    # eval_dl chỉ chứa 1 modality (đúng với STUDENT_MODAL_IDX)
    train_auc = compute_id_auc(train_dl, student, device,
                               modal_idx=STUDENT_MODAL_IDX)  # truyền idx nếu loader trả [B,2,...]
    test_auc  = compute_id_auc(test_dl,  student, device,
                               modal_idx=STUDENT_MODAL_IDX)

    train_metrics = {
        'loss':             train_loss,
        'loss_task':        train_task,
        'loss_kd':          train_kd,
        'loss_rkd_d':       train_rkd_d,
        'loss_rkd_a':       train_rkd_a,
        'auc_id_cosine':    train_auc['id_cosine'],
        'auc_id_euclidean': train_auc['id_euclidean'],
    }
    test_metrics = {
        'auc_id_cosine':    test_auc['id_cosine'],
    }

    writer.add_scalar('Loss/total',   train_loss,   epoch + 1)
    writer.add_scalar('Loss/task',    train_task,   epoch + 1)
    writer.add_scalar('Loss/kd',      train_kd,     epoch + 1)
    writer.add_scalar('Loss/rkd_d',   train_rkd_d,  epoch + 1)
    writer.add_scalar('Loss/rkd_a',   train_rkd_a,  epoch + 1)
    writer.add_scalars('AUC/cosine',
        {'train': train_auc['id_cosine'],    'test': test_auc['id_cosine']},    epoch + 1)
    writer.add_scalars('AUC/euclidean',
        {'train': train_auc['id_euclidean'], 'test': test_auc['id_euclidean']}, epoch + 1)

    display_metrics(epoch + 1, train_metrics, test_metrics)
    manager.log_metrics(epoch + 1, {**train_metrics, **test_metrics})

    ckpt_saver(student, optimizer, epoch + 1, test_metrics, scheduler)
    early_stopping(test_metrics, student, epoch + 1)
    scheduler.step(epoch)

    if early_stopping.early_stop:
        manager.log_text('Early stopping triggered.')
        break

writer.close()
manager.log_text('new CROSS-MODAL KNOWLEDGE DISTILLATION HOAN TAT.')

BAT DAU new CROSS-MODAL KNOWLEDGE DISTILLATION

--- Epoch 1/100 ---

Ep 1:
╒══════════════════╤═════════╤════════╕
│ Metric           │   Train │ Test   │
╞══════════════════╪═════════╪════════╡
│ loss             │ 43.5817 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_task        │ 29.4758 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_kd          │  0.8509 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_rkd_d       │  0.0047 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_rkd_a       │  0.0044 │ -      │
├──────────────────┼─────────┼────────┤
│ auc_id_cosine    │  0.93   │ 0.8882 │
├──────────────────┼─────────┼────────┤
│ auc_id_euclidean │  0.93   │ -      │
╘══════════════════╧═════════╧════════╛
Ep 1: loss: 43.5817, loss_task: 29.4758, loss_kd: 0.8509, loss_rkd_d: 0.0047, loss_rkd_a: 0.0044, auc_id_cosine: 0.8882, auc_id_euclidean: 0.9300
--> SAVE BEST MODEL (auc_id_cosine: 0.8882)

--- Epoch 2/100 ---

Ep 2:
╒══════════════════╤═══════

## 9. Resume Training từ Checkpoint

> Chạy khi Colab disconnect.  
> **Cách dùng:** Chạy cell Setup → Imports → Data → Teacher → Student → Projector → Loss → Setup Train,  
> sau đó chạy cell này, rồi chạy lại cell fit.

In [ ]:
CKPT_PATH = os.path.join(manager.ckpt_dir, 'best_model.pth')

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'Không tìm thấy checkpoint: {CKPT_PATH}')

checkpoint = torch.load(CKPT_PATH, map_location=device,weights_only=False)
student.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
if 'scheduler_state_dict' in checkpoint:
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
if 'projector_state_dict' in checkpoint:
    projector.load_state_dict(checkpoint['projector_state_dict'])
    print('Projector state loaded.')
else:
    print('Không tìm thấy projector_state_dict — projector khởi tạo ngẫu nhiên.')

START_EPOCH = checkpoint['epoch']
print(f'Resume từ epoch {START_EPOCH} — chạy lại cell "cell-fit" để tiếp tục.')

Không tìm thấy projector_state_dict — projector khởi tạo ngẫu nhiên.
Resume từ epoch 55 — chạy lại cell "cell-fit" để tiếp tục.


## 10. Đánh giá Final

So sánh:
- **Teacher (single modal baseline):** ConvNeXt backbone dùng đúng modality đó, không có cross-modal
- **Student (cross-modal KD):** MobileNetV3 học từ fused teacher 1024-D

In [ ]:
best_ckpt_path = os.path.join(manager.ckpt_dir, 'best_model.pth')
best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
student.load_state_dict(best_ckpt['model_state_dict'])
student.eval()

student_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, student, device)
student_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, student, device)

teacher_gp_auc = {
    'id_cosine': 0.9761,
    'id_euclidean': 0.9761,
    'rank_1':0.8021
}
teacher_concat_auc = {
    'id_cosine': 0.9825,
    'id_euclidean': 0.9825,
    'rank_1':0.8403
}


# 3 cột: ConvNeXt single-modal | Full ConcatTeacher | Student sau KD
compare_rows = [
    ['Model',
     f"{CONFIGURATION['teacher_backbone']} ({modal_name} only)",
     f"ConcatTeacher ({CONFIGURATION['teacher_backbone']} ×2)",
     f"{CONFIGURATION['backbone']} (cross-modal KD)"],
    ['Input',
     f'{modal_name} only',
     'albedo + normalmap',
     f'{modal_name} only'],
    ['Params',
     f'~{teacher_params // 2 // 1_000_000}M (est.)',
     f'~{teacher_params // 1_000_000}M',
     f'~{total_p // 1_000_000}M'],
    ['Cosine AUC (gallery→probe)',
     f"{teacher_gp_auc['id_cosine']:.4f}",
     f"{teacher_concat_auc['id_cosine']:.4f}",
     f"{student_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)',
     f"{teacher_gp_auc['id_euclidean']:.4f}",
     f"{teacher_concat_auc['id_euclidean']:.4f}",
     f"{student_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc (gallery→probe)',
     f"{teacher_gp_auc['rank_1']:.4f}",
     f"{teacher_concat_auc['rank_1']:.4f}",
     f"{student_gp_rank1:.4f}"],
]
print('\n--- So sánh kết quả ---')
print(tabulate(compare_rows,
               headers=['Metric', 'Single-modal baseline', 'Upper bound (2 modal)', 'Student (cross-modal KD)'],
               tablefmt='fancy_grid'))


--- So sánh kết quả ---
╒═══════════════════════════════╤═══════════════════════════════╤════════════════════════════════════╤════════════════════════════════════════╕
│ Metric                        │ Single-modal baseline         │ Upper bound (2 modal)              │ Student (cross-modal KD)               │
╞═══════════════════════════════╪═══════════════════════════════╪════════════════════════════════════╪════════════════════════════════════════╡
│ Model                         │ convnextv2_tiny (albedo only) │ ConcatTeacher (convnextv2_tiny ×2) │ mobilenetv3_large_100 (cross-modal KD) │
├───────────────────────────────┼───────────────────────────────┼────────────────────────────────────┼────────────────────────────────────────┤
│ Input                         │ albedo only                   │ albedo + normalmap                 │ albedo only                            │
├───────────────────────────────┼───────────────────────────────┼────────────────────────────────────┼─────────

In [ ]:
# best_ckpt_path = os.path.join(manager.ckpt_dir, 'best_model.pth')
# best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
# student.load_state_dict(best_ckpt['model_state_dict'])
# student.eval()
# print(f"Best model từ epoch {best_ckpt.get('epoch', '?')}")

# student_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, student, device)
# student_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, student, device)

# # 3 cột: ConvNeXt single-modal | Full ConcatTeacher | Student sau KD
# compare_rows = [
#     ['Model',
#      f"{CONFIGURATION['teacher_backbone']} ({modal_name} only)",
#      f"ConcatTeacher ({CONFIGURATION['teacher_backbone']} ×2)",
#      f"{CONFIGURATION['backbone']} (cross-modal KD)"],
#     ['Input',
#      f'{modal_name} only',
#      'albedo + normalmap',
#      f'{modal_name} only'],
#     ['Params',
#      f'~{teacher_params // 2 // 1_000_000}M (est.)',
#      f'~{teacher_params // 1_000_000}M',
#      f'~{total_p // 1_000_000}M'],
#     ['Cosine AUC (gallery→probe)',
#      f"{teacher_gp_auc['id_cosine']:.4f}",
#      f"{teacher_concat_auc['id_cosine']:.4f}",
#      f"{student_gp_auc['id_cosine']:.4f}"],
#     ['Euclidean AUC (gallery→probe)',
#      f"{teacher_gp_auc['id_euclidean']:.4f}",
#      f"{teacher_concat_auc['id_euclidean']:.4f}",
#      f"{student_gp_auc['id_euclidean']:.4f}"],
#     ['Rank-1 Acc (gallery→probe)',
#      f"{teacher_gp_rank1:.4f}",
#      f"{teacher_concat_rank1:.4f}",
#      f"{student_gp_rank1:.4f}"],
# ]
# print('\n--- So sánh kết quả ---')
# print(tabulate(compare_rows,
#                headers=['Metric', 'Single-modal baseline', 'Upper bound (2 modal)', 'Student (cross-modal KD)'],
#                tablefmt='fancy_grid'))

## 11. Export ONNX

Export phần inference (backbone + embedding + L2 normalize, bỏ MagLinear và projector)  
để chuẩn bị quantize INT8 deploy edge device.

In [ ]:
import sys
!{sys.executable} -m pip install onnxscript

class InferenceWrapper(nn.Module):
  """Backbone + embedding + L2 normalize — không có MagLinear, không có Projector."""
  def __init__(self, model):
    super().__init__()
    self.backbone  = model.backbone
    self.embedding = model.embedding

  def forward(self, x):
    emb = self.embedding(self.backbone(x))
    return F.normalize(emb, p=2, dim=1)


inference_model = InferenceWrapper(student).eval().cpu()
dummy_input = torch.randn(1, 3, 112, 112)

ONNX_PATH = os.path.join(manager.ckpt_dir, 'kd_crossmodal_mobilenetv3_fr.onnx')

torch.onnx.export(
    inference_model,
    dummy_input,
    ONNX_PATH,
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17,
)
print(f'Exported ONNX: {ONNX_PATH}')
print(f'Input : [B, 3, 112, 112] — chỉ cần {modal_name}')
print(f'Output: [B, 512] — L2-normalized embedding')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 135.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 27.3 MB/s eta 0:00:00


W0611 22:52:42.021000 7142 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `InferenceWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `InferenceWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:68: adapt: Asserti

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Exported ONNX: /content/drive/MyDrive/experiments/(1-15-100-200)new_lan2_KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo/checkpoints/kd_crossmodal_mobilenetv3_fr.onnx
Input : [B, 3, 112, 112] — chỉ cần albedo
Output: [B, 512] — L2-normalized embedding


In [ ]:
print('Merging external weights into single ONNX file...')
model_proto = onnx.load(ONNX_PATH, load_external_data=True)

ONNX_PATH_MERGED = ONNX_PATH.replace('.onnx', '_merged.onnx')
onnx.save_model(model_proto, ONNX_PATH_MERGED, save_as_external_data=False)

print(f'Done: {ONNX_PATH_MERGED}')
print(f'Size: {os.path.getsize(ONNX_PATH_MERGED) / 1024 / 1024:.1f} MB')

Merging external weights into single ONNX file...
Done: /content/drive/MyDrive/experiments/(1-15-100-200)new_lan2_KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo/checkpoints/kd_crossmodal_mobilenetv3_fr_merged.onnx
Size: 13.6 MB
